# ELECTRA + ScalarMix — DeepSeek synthetic train → OOD eval

Train on **DeepSeek synthetic Q/A** from Google Drive (no DANN). Evaluate on **OneStop English** and **RACE middle / high** loaded fresh from Hugging Face with **original corpus labels** (not LLM-judge labels).

Outputs (model, metrics JSON, summary CSV, confusion matrices) are saved under `DRIVE_OUT_DIR`.

In [ ]:
!pip install -q transformers datasets scikit-learn torch pandas matplotlib seaborn tqdm

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ── Paths (edit to match your Drive layout) ──────────────────────────────
DRIVE_SYNTH_CSV = "/content/drive/MyDrive/multi_corpus/synthetic_deepseek_qa.csv"
DRIVE_OUT_DIR = "/content/drive/MyDrive/beyond_flesch/trail/electra_deepseek"

MODEL_NAME = "google/electra-large-discriminator"
MAX_LEN = 512
BATCH_SIZE = 8
EPOCHS = 3
LR = 1e-5
WARMUP_STEPS = 50
VAL_FRAC = 0.20
RNG_SEED = 42
LABEL_SMOOTHING = 0.05

label2id = {"elementary": 0, "middle": 1, "high": 2}
id2label = {v: k for k, v in label2id.items()}
LABEL_NAMES = ["elementary", "middle", "high"]

In [ ]:
from __future__ import annotations

import json
import os
import random
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from datasets import load_dataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from torch.nn import Parameter, ParameterList
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup

os.makedirs(DRIVE_OUT_DIR, exist_ok=True)
(Path(DRIVE_OUT_DIR) / "confusion_matrices").mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RNG_SEED)
print("Device:", device)
print("Output:", DRIVE_OUT_DIR)

In [ ]:
# ── ScalarMix (inlined, no allennlp) ─────────────────────────────────────
class ScalarMix(nn.Module):
    def __init__(self, mixture_size: int, trainable: bool = True) -> None:
        super().__init__()
        self.scalar_parameters = ParameterList(
            [Parameter(torch.zeros(1), requires_grad=trainable) for _ in range(mixture_size)]
        )
        self.gamma = Parameter(torch.ones(1), requires_grad=trainable)

    def forward(self, tensors: List[torch.Tensor]) -> torch.Tensor:
        w = torch.nn.functional.softmax(
            torch.cat([p for p in self.scalar_parameters]), dim=0
        )
        w = torch.split(w, 1)
        return self.gamma * sum(weight * t for weight, t in zip(w, tensors))


class ElectraScalarMixClassifier(nn.Module):
    """ELECTRA + ScalarMix + mean-pool → 3-class head (no domain adversarial)."""

    def __init__(self, model_name: str, num_classes: int = 3, dropout: float = 0.2) -> None:
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = int(self.encoder.config.hidden_size)
        n_layers = int(self.encoder.config.num_hidden_layers) + 1
        self.scalar_mix = ScalarMix(n_layers)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_classes),
        )

    def encode_pooled(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        out = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )
        mixed = self.dropout(self.scalar_mix(list(out.hidden_states)))
        mask = attention_mask.unsqueeze(-1).float()
        summed = (mixed * mask).sum(dim=1)
        return summed / mask.sum(dim=1).clamp(min=1e-9)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        return self.head(self.encode_pooled(input_ids, attention_mask))

print("Model classes OK")

In [ ]:
# ── Load DeepSeek synthetic CSV from Drive ───────────────────────────────
if not os.path.exists(DRIVE_SYNTH_CSV):
    alt = os.path.basename(DRIVE_SYNTH_CSV)
    if os.path.exists(alt):
        DRIVE_SYNTH_CSV = alt
    else:
        raise FileNotFoundError(f"Synthetic CSV not found: {DRIVE_SYNTH_CSV}")

synth = pd.read_csv(DRIVE_SYNTH_CSV)
synth.columns = [c.strip().lower() for c in synth.columns]
for col in ("question", "answer", "grade_level"):
    if col not in synth.columns:
        raise ValueError(f"Expected column '{col}' in {DRIVE_SYNTH_CSV}, got {list(synth.columns)}")

synth = synth.dropna(subset=["question", "answer", "grade_level"]).reset_index(drop=True)
synth["grade_level"] = synth["grade_level"].astype(str).str.strip().str.lower()
synth = synth[synth["grade_level"].isin(label2id)].reset_index(drop=True)
synth["full_text"] = (
    "Question: " + synth["question"].astype(str) + "\n"
    "Answer: " + synth["answer"].astype(str)
)
synth["label"] = synth["grade_level"].map(label2id).astype(int)

print(f"Synthetic rows: {len(synth)}")
print(synth["grade_level"].value_counts())

X = synth["full_text"].tolist()
y = synth["label"].values
X_tr, X_va, y_tr, y_va = train_test_split(
    X, y, test_size=VAL_FRAC, stratify=y, random_state=RNG_SEED
)
print(f"Train {len(X_tr)} | Val {len(X_va)}")

In [ ]:
# ── OOD corpora from Hugging Face (original labels) ──────────────────────

def load_onestop_hf() -> Tuple[List[str], np.ndarray]:
  """SetFit/onestop_english — label 0/1/2 → elementary/middle/high."""
  ds = load_dataset("SetFit/onestop_english")
  texts, labels = [], []
  mapping = {0: 0, 1: 1, 2: 2}
  for split in ds:
    for r in ds[split]:
      li = int(r.get("label", -1))
      if li not in mapping:
        continue
      text = (r.get("text") or r.get("content") or "").strip()
      if text:
        texts.append(text)
        labels.append(mapping[li])
  return texts, np.array(labels, dtype=int)


def load_race_hf(config: str) -> Tuple[List[str], np.ndarray]:
  """ehovy/race middle|high — label is the config name."""
  assert config in ("middle", "high")
  ds = load_dataset("ehovy/race", config)
  level_id = label2id[config]
  texts, labels, seen = [], [], set()
  for split in ds:
    for r in ds[split]:
      article = (r.get("article") or "").strip()
      if not article or article in seen:
        continue
      seen.add(article)
      texts.append(article)
      labels.append(level_id)
  return texts, np.array(labels, dtype=int)


print("Loading OneStop English …")
OSE_X, OSE_y = load_onestop_hf()
print(f"  OneStop: n={len(OSE_X)}  class counts={np.bincount(OSE_y, minlength=3)}")

print("Loading RACE middle …")
RACE_M_X, RACE_M_y = load_race_hf("middle")
print(f"  RACE-middle: n={len(RACE_M_X)}")

print("Loading RACE high …")
RACE_H_X, RACE_H_y = load_race_hf("high")
print(f"  RACE-high: n={len(RACE_H_X)}")

In [ ]:
# ── Dataset / DataLoader ───────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class TextClsDataset(Dataset):
    def __init__(self, texts: List[str], labels: np.ndarray) -> None:
        self.texts = texts
        self.labels = labels.astype(int)

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=MAX_LEN,
            padding="max_length",
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }


def make_loader(texts, labels, shuffle: bool) -> DataLoader:
    return DataLoader(
        TextClsDataset(texts, labels),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
    )

train_loader = make_loader(X_tr, y_tr, shuffle=True)
val_loader = make_loader(X_va, y_va, shuffle=False)

In [ ]:
# ── Train ──────────────────────────────────────────────────────────────────
model = ElectraScalarMixClassifier(MODEL_NAME).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
total_steps = EPOCHS * len(train_loader)
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=WARMUP_STEPS, num_training_steps=total_steps
)

history = []
best_val_f1 = -1.0
best_path = os.path.join(DRIVE_OUT_DIR, "best_model.pt")


@torch.no_grad()
def eval_loader(loader: DataLoader) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    ys, preds = [], []
    for batch in loader:
        logits = model(
            batch["input_ids"].to(device),
            batch["attention_mask"].to(device),
        )
        preds.extend(logits.argmax(dim=1).cpu().numpy().tolist())
        ys.extend(batch["label"].numpy().tolist())
    return np.array(ys), np.array(preds)


for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        optimizer.zero_grad()
        logits = model(
            batch["input_ids"].to(device),
            batch["attention_mask"].to(device),
        )
        loss = criterion(logits, batch["label"].to(device))
        loss.backward()
        optimizer.step()
        scheduler.step()
        running += loss.item()
    y_v, p_v = eval_loader(val_loader)
    val_f1 = f1_score(y_v, p_v, average="macro", zero_division=0)
    val_acc = accuracy_score(y_v, p_v)
    avg_loss = running / max(len(train_loader), 1)
    print(f"Epoch {epoch+1}: loss={avg_loss:.4f} val_acc={val_acc:.4f} val_macro_f1={val_f1:.4f}")
    history.append({"epoch": epoch + 1, "train_loss": avg_loss, "val_acc": val_acc, "val_macro_f1": val_f1})
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), best_path)
        print(f"  → saved best checkpoint (val macro-F1 {val_f1:.4f})")

model.load_state_dict(torch.load(best_path, map_location=device))
print("Training done. Best val macro-F1:", best_val_f1)

In [ ]:
# ── Evaluate OOD + confusion matrices ────────────────────────────────────
CM_DIR = os.path.join(DRIVE_OUT_DIR, "confusion_matrices")


def save_confusion_matrix(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    title: str,
    out_path: str,
    labels: List[int],
) -> None:
    names = [id2label[i] for i in labels]
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=names,
        yticklabels=names,
        ax=ax,
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


@torch.no_grad()
def predict_texts(texts: List[str], batch_size: int = 16) -> np.ndarray:
    model.eval()
    preds = []
    for i in tqdm(range(0, len(texts), batch_size), desc="predict", leave=False):
        chunk = texts[i : i + batch_size]
        enc = tokenizer(
            chunk,
            truncation=True,
            max_length=MAX_LEN,
            padding=True,
            return_tensors="pt",
        )
        logits = model(enc["input_ids"].to(device), enc["attention_mask"].to(device))
        preds.extend(logits.argmax(dim=1).cpu().numpy().tolist())
    return np.array(preds, dtype=int)


def eval_corpus(name: str, texts: List[str], y_true: np.ndarray, labels: List[int]) -> Dict:
    y_pred = predict_texts(texts)
    acc = accuracy_score(y_true, y_pred)
    f1m = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    report = classification_report(
        y_true,
        y_pred,
        labels=labels,
        target_names=[id2label[i] for i in labels],
        zero_division=0,
    )
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    print(report)
    print(f"Accuracy: {acc:.4f}  Macro-F1 (labels={labels}): {f1m:.4f}")
    cm_path = os.path.join(CM_DIR, f"cm_{name.replace(' ', '_').lower()}.png")
    save_confusion_matrix(y_true, y_pred, name, cm_path, labels)
    print(f"Saved CM → {cm_path}")
    return {
        "corpus": name,
        "n": int(len(y_true)),
        "accuracy": float(acc),
        "macro_f1": float(f1m),
        "labels_evaluated": labels,
        "classification_report": report,
        "confusion_matrix_png": cm_path,
    }


results = {"config": {
    "model": MODEL_NAME,
    "synth_csv": DRIVE_SYNTH_CSV,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "best_val_macro_f1": float(best_val_f1),
    "train_history": history,
}, "evaluations": []}

y_va_pred = predict_texts(X_va)
results["evaluations"].append(
    eval_corpus("synthetic_val", X_va, y_va, labels=[0, 1, 2])
)
results["evaluations"].append(
    eval_corpus("onestop_english", OSE_X, OSE_y, labels=[0, 1, 2])
)
# RACE: gold is only middle or high; report macro-F1 on present class + 3x3 CM
results["evaluations"].append(
    eval_corpus("race_middle", RACE_M_X, RACE_M_y, labels=[0, 1, 2])
)
results["evaluations"].append(
    eval_corpus("race_high", RACE_H_X, RACE_H_y, labels=[0, 1, 2])
)

summary = pd.DataFrame([
    {"corpus": e["corpus"], "n": e["n"], "accuracy": e["accuracy"], "macro_f1": e["macro_f1"]}
    for e in results["evaluations"]
])
print("\nSummary:\n", summary.to_string(index=False))

json_path = os.path.join(DRIVE_OUT_DIR, "eval_results.json")
csv_path = os.path.join(DRIVE_OUT_DIR, "eval_summary.csv")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
summary.to_csv(csv_path, index=False)
print(f"\nSaved JSON → {json_path}")
print(f"Saved CSV  → {csv_path}")
print(f"Model      → {best_path}")
print(f"CMs        → {CM_DIR}")